In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# import stargazer
from stargazer.stargazer import Stargazer

In [ ]:
import sys
import os

# Add parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
sys.path.append(parent_dir)

In [ ]:
from package_files.benefits_defns import *

# Read in data

In [ ]:
df = pd.read_csv('../exports/occ_year_coeff_analysis.csv')

In [ ]:
df

In [ ]:
df['DURATION_CALC_ai'] = pd.to_timedelta(df['DURATION_CALC_ai']).dt.total_seconds() / (24 * 3600)

In [ ]:
df['DURATION_CALC_non_ai'] = pd.to_timedelta(df['DURATION_CALC_non_ai']).dt.total_seconds() / (24 * 3600)

In [ ]:
df['DURATION_CALC_ai']

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
# show rows in df with nas in ai coefficient
df[df['AI Coefficient'].isna()]

## Add dfs to dict

In [ ]:
# create dict of benefit dfs
df_dict = {}
for benefit in benefits4:
    benefit_df = df[df['Benefit'] == benefit]
    df_dict[benefit] = benefit_df

In [ ]:
plt.hist(df_dict['EDU_ASSISTANCE']['AI Demand'], bins=20)

# Run Models

## Baseline with Demand (1)

In [ ]:
import statsmodels.api as sm
benefit_models = {}
for benefit in df_dict.keys():
    benefit_df = df_dict[benefit]
    # drop nas in AI Coefficient
    benefit_df = benefit_df.dropna(subset=['AI Coefficient'])
    # Define the dependent and independent variables
    X = benefit_df['AI Demand']
    y = benefit_df['AI Coefficient']

    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    benefit_models[benefit] = []
    benefit_models[benefit].append(model)
    # Print the summary of the regression
    print(model.summary())

## + Prevalence (2)

In [ ]:
import statsmodels.api as sm
for benefit in benefits4:
    benefit_df = df_dict[benefit]
    # drop nas in AI Coefficient
    benefit_df = benefit_df.dropna(subset=['AI Coefficient'])
    # Define the dependent and independent variables
    # X is AI ROLE % and Percent_with_{benefit}
    # benefit_df.rename(columns={f'{benefit}_prevalence}': f'Prevalence: {label}', inplace=True)
    label = benefits_labels_map[benefit]
    X = benefit_df[['AI Demand', f'Prevalence: {label}']]
    y = benefit_df['AI Coefficient']

    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    benefit_models[benefit].append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
from IPython.core.display import HTML

## + Salary (3)

In [ ]:
import statsmodels.api as sm
for benefit in benefits4:
    benefit_df = df_dict[benefit]
    # drop nas in AI Coefficient
    benefit_df = benefit_df.dropna(subset=['AI Coefficient'])
    # Define the dependent and independent variables
    # X is AI ROLE % and Percent_with_{benefit}
    label = benefits_labels_map[benefit]
    # benefit_df.rename(columns={f'{benefit}_prevalence}': f'Prevalence: {label}', inplace=True)
    X = benefit_df[['AI Demand', f'Prevalence: {label}', 'Salary (Log) Premium']]
    y = benefit_df['AI Coefficient']

    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    benefit_models[benefit].append(model)
    # Print the summary of the regression
    print(model.summary())

In [ ]:
import numpy as np

## % Change Model (4)

In [ ]:
import statsmodels.api as sm
for benefit in benefits4:
    benefit_df = df_dict[benefit]
    # drop nas in AI Coefficient
    benefit_df = benefit_df.replace([np.inf, -np.inf], np.nan)  # Replace inf with NaN

    benefit_df = benefit_df.dropna(subset=['AI Coefficient', 'AI Demand % Change'])
    
    # Define the dependent and independent variables
    # X is AI ROLE % and Percent_with_{benefit}
    label = benefits_labels_map[benefit]
    # benefit_df.rename(columns={f'{benefit}_prevalence}': f'Prevalence: {label}', inplace=True)
    X = benefit_df['AI Demand % Change']
    print(len(X))
    y = benefit_df['AI Coefficient']
    print(len(y))

    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    benefit_models[benefit].append(model)
    # Print the summary of the regression
    print(model.summary())

## Duration Model (5)

In [ ]:
import statsmodels.api as sm
for benefit in benefits4:
    benefit_df = df_dict[benefit]
    # drop nas in AI Coefficient
    benefit_df = benefit_df.replace([np.inf, -np.inf], np.nan)  # Replace inf with NaN

    benefit_df = benefit_df.dropna(subset=['AI Coefficient', 'DURATION_CALC_ai'])
    benefit_df.rename(columns={'DURATION_CALC_ai': 'Avg AI Posting Duration'}, inplace=True)
    # Define the dependent and independent variables
    # X is AI ROLE % and Percent_with_{benefit}
    label = benefits_labels_map[benefit]
    # benefit_df.rename(columns={f'{benefit}_prevalence}': f'Prevalence: {label}', inplace=True)
    X = benefit_df['Avg AI Posting Duration']
    print(len(X))
    print(X.dtypes)
    y = benefit_df['AI Coefficient']
    print(len(y))

    # Add a constant to the independent variable
    X = sm.add_constant(X)

    # Fit the OLS model
    model = sm.OLS(y, X).fit()
    print(benefit)
    benefit_models[benefit].append(model)
    # Print the summary of the regression
    print(model.summary())

# Display Models

In [ ]:
for benefit in benefits4:
    stargazer = Stargazer(benefit_models[benefit])
    stargazer.title(f'Coefficients for {benefits_labels_map[benefit]}')
    display(HTML(stargazer.render_html()))
    # export to latex
    with open(f'../exports/occ_year_coeff_{benefit}.tex', 'w') as f:
        f.write(stargazer.render_latex())

# Testing Assumptions

## Linearity of Relationships

In [ ]:
import matplotlib.pyplot as plt

for benefit in benefits4:
    model = benefit_models[benefit][0]  # Adjust to the model you want to test
    plt.scatter(model.fittedvalues, model.resid)
    plt.axhline(y=0, color='r', linestyle='--')
    plt.title(f'Residual Plot for {benefit}')
    plt.xlabel('Fitted Values')
    plt.ylabel('Residuals')
    plt.show()


## Multicollinearity

In [ ]:

from statsmodels.stats.outliers_influence import variance_inflation_factor

for benefit in benefits4:
    benefit_df = df_dict[benefit]
    
    # Select the independent variables for VIF calculation
    X = benefit_df[['AI Demand % Change', f'Prevalence: {benefits_labels_map[benefit]}', 'Salary (Log) Premium']]
    
    # Replace inf/-inf with NaN
    X = X.replace([np.inf, -np.inf], np.nan)
    
    # Drop rows with NaN values in any of the selected columns
    X = X.dropna()
    
    # Add a constant to the independent variables
    X = sm.add_constant(X)
    
    # Check if there is sufficient data for VIF calculation after dropping NaNs
    if X.shape[0] > 0:
        # Calculate VIF for each variable
        vif_data = pd.DataFrame()
        vif_data["feature"] = X.columns
        vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        print(f'VIF for {benefit}:')
        print(vif_data)
    else:
        print(f"Insufficient data for VIF calculation in {benefit} after cleaning.")


## Autocorrelation of Errors

In [ ]:
from statsmodels.stats.stattools import durbin_watson

for benefit in benefits4:
    model = benefit_models[benefit][0]  
    dw_stat = durbin_watson(model.resid)
    print(f'Durbin-Watson statistic for {benefit}: {dw_stat}')


## Normality of Residuals

In [ ]:
import scipy.stats as stats

for benefit in benefits4:
    print(benefit)
    for i in range(5):
        print("Model ", i+1)
        model = benefit_models[benefit][i]  # change for model of interest
        
        # QQ-plot
        stats.probplot(model.resid, dist="norm", plot=plt)
        plt.title(f'QQ-Plot for {benefit}')
        plt.show()
        
        # Jarque-Bera test
        jb_stat, jb_pvalue = stats.jarque_bera(model.resid)
    print(f'Jarque-Bera test for {benefit}: JB Stat={jb_stat}, p-value={jb_pvalue}')


## Homoscedasticity 

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
for benefit in benefits4:
    print(f'Breusch-Pagan test for {benefit}:')
    for i in range(4):
        model = benefit_models[benefit][i]  # Use the relevant model
        test_result = het_breuschpagan(model.resid, model.model.exog)
        print(f'LM Statistic: {test_result[i]}, p-value: {test_result[1]}')


# Outliers

In [ ]:
for benefit in benefits4:
    print(benefit)
    benefit_df = df_dict[benefit]
    # Check for outliers in the 'AI Coefficient' column using the IQR method
    Q1 = benefit_df['AI Coefficient'].quantile(0.25)
    Q3 = benefit_df['AI Coefficient'].quantile(0.75)
    IQR = Q3 - Q1

    # Define the bounds for outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Find outliers
    outliers = benefit_df[(benefit_df['AI Coefficient'] < lower_bound) | (benefit_df['AI Coefficient'] > upper_bound)]
    display(outliers)



In [ ]:
outliers

# Correlations

In [ ]:
df_single = df[df['Benefit'] == 'EDU_ASSISTANCE']

In [ ]:
# select all columns beginning with Prevalence: 
prevalence_cols = [col for col in df_single.columns if col.startswith('Prevalence: ')]
df_prevalence = df_single[prevalence_cols + ['MEAN SALARY']]

In [ ]:
correlation_matrix = df_prevalence.corr()

In [ ]:
correlation_matrix

In [ ]:
import seaborn as sns
plt.figure(figsize=(8,6))
sns.heatmap(correlation_matrix, annot=True, cmap='Blues', cbar = True)
plt.xticks(ticks=np.arange(len(correlation_matrix.columns))+0.5, labels=benefits4_labels + ['Mean Salary'], rotation=45, ha='right')
plt.yticks(ticks=np.arange(len(correlation_matrix.columns))+0.5, labels=benefits4_labels + ['Mean Salary'], rotation=0, va='center')
plt.figtext(0.5, -0.15, 'Correlation Matrix of Benefits Prevalence on Occupation-Year Level', wrap=True, horizontalalignment='right', fontsize=10)